# Comparing 11 Machine Learning Models

**Goal:** predict 30-day readmission (`readmitted_30`: 1 = yes, 0 = no) with 11 different models, compare them fairly, and pick the best one.

Every model gets **exactly the same** data, the same patient-level train/test split and the same preparation steps, so the comparison is fair.

**Main score: ROC-AUC on the test set.** It measures how well a model ranks risky patients above safe ones (0.5 = guessing, 1.0 = perfect). We use it instead of accuracy because only 11% of patients are readmitted, so accuracy is misleading.

## 0. Install the extra libraries (only needed once)
Run this, then click **Restart** and **Run All**.

In [ ]:
# CatBoost is on its own line: if it fails to install, the others still install
%pip install scikit-learn xgboost lightgbm
%pip install catboost

## 1. Load the clean data from MySQL

In [ ]:
import sys, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
sys.path.insert(0, str(Path.cwd().parent))
from config import get_engine

df = pd.read_sql("SELECT * FROM encounters_clean", get_engine())
print(df.shape)

## 2. Features and target (same as ml_model.ipynb)

In [ ]:
target = "readmitted_30"
drop_cols = ["encounter_id", "patient_nbr", "readmitted", "readmit_status",
             "readmitted_30", "a1c_tested"]

X = df.drop(columns=drop_cols)
y = df[target]
groups = df["patient_nbr"]

num_cols = X.select_dtypes(include="number").columns.tolist()
cat_cols = X.select_dtypes(exclude="number").columns.tolist()
X[cat_cols] = X[cat_cols].fillna("Unknown")

## 3. Split by patient (80% train, 20% test)

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
print("Train:", X_train.shape, "| Test:", X_test.shape)

## 4. Define all the models

| Family | Models |
|---|---|
| Straight-line models | Linear Regression, Logistic Regression |
| Tree models | Decision Tree, Random Forest |
| Boosting models (trees that learn from each other's mistakes) | AdaBoost, Gradient Boosting, XGBoost, LightGBM, CatBoost |
| Distance / boundary models | KNN, SVM |

Notes:
- **Decision Tree** is limited to 6 levels (`max_depth=6`). We also train one with **no limit** to show what overfitting looks like.
- **SVM** is trained on a random 10,000-patient sample, because on all 80,000 patients it takes far too long.
- **KNN** is fast to train but slow to predict (it compares each test patient with every training patient). Expect it to take a few minutes.
- If XGBoost, LightGBM or CatBoost isn't installed, that model is skipped instead of crashing.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (RandomForestClassifier, AdaBoostClassifier,
                              GradientBoostingClassifier)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

def make_pipeline(model):
    prep = ColumnTransformer([
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ])
    return Pipeline([("prep", prep), ("model", model)])

models = {
    "Linear Regression": LinearRegression(),
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(max_depth=6, random_state=42),
    "Decision Tree (no limit)": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=200, min_samples_leaf=20,
                                            n_jobs=-1, random_state=42),
    "AdaBoost": AdaBoostClassifier(n_estimators=200, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
}

try:
    from xgboost import XGBClassifier
    models["XGBoost"] = XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=5,
                                      random_state=42, n_jobs=-1)
except ImportError:
    print("XGBoost not installed - skipped")

try:
    from lightgbm import LGBMClassifier
    models["LightGBM"] = LGBMClassifier(n_estimators=300, learning_rate=0.05,
                                        random_state=42, verbose=-1)
except ImportError:
    print("LightGBM not installed - skipped")

try:
    from catboost import CatBoostClassifier
    models["CatBoost"] = CatBoostClassifier(iterations=300, learning_rate=0.05, depth=6,
                                            random_state=42, verbose=0)
except ImportError:
    print("CatBoost not installed - skipped")

models["KNN"] = KNeighborsClassifier(n_neighbors=50, n_jobs=-1)
models["SVM (10k sample)"] = SVC(random_state=42)

print(len(models), "models ready:", list(models))

## 5. Train and test every model
For each model we record:
- **Train AUC** and **Test AUC**. A big gap between them means **overfitting** (the model memorised the training patients instead of learning general patterns).
- **Accuracy, Recall, F1** at the default cutoff.
- **Training time** and **prediction time** in seconds.

In [ ]:
from sklearn.metrics import roc_auc_score, accuracy_score, recall_score, f1_score

rows = []
for name, model in models.items():
    pipe = make_pipeline(model)
    X_fit, y_fit = X_train, y_train
    if name.startswith("SVM"):                       # SVM is too slow on all patients
        sample = np.random.RandomState(42).choice(len(X_train), 10_000, replace=False)
        X_fit, y_fit = X_train.iloc[sample], y_train.iloc[sample]

    start = time.time()
    pipe.fit(X_fit, y_fit)
    fit_seconds = time.time() - start

    start = time.time()
    if name == "Linear Regression":                  # gives any number
        test_score, train_score = pipe.predict(X_test), pipe.predict(X_fit)
        test_pred = (test_score >= 0.5).astype(int)
    elif name.startswith("SVM"):                     # gives a distance from the boundary
        test_score, train_score = pipe.decision_function(X_test), pipe.decision_function(X_fit)
        test_pred = (test_score >= 0).astype(int)
    else:                                            # gives a probability 0 to 1
        test_score = pipe.predict_proba(X_test)[:, 1]
        train_score = pipe.predict_proba(X_fit)[:, 1]
        test_pred = (test_score >= 0.5).astype(int)
    predict_seconds = time.time() - start

    rows.append({
        "Model": name,
        "Train AUC": roc_auc_score(y_fit, train_score),
        "Test AUC": roc_auc_score(y_test, test_score),
        "Accuracy": accuracy_score(y_test, test_pred),
        "Recall": recall_score(y_test, test_pred),
        "F1": f1_score(y_test, test_pred),
        "Train seconds": fit_seconds,
        "Predict seconds": predict_seconds,
    })
    print(f"{name:26s} Test AUC {rows[-1]['Test AUC']:.3f}   ({fit_seconds:.1f}s train)")

results = (pd.DataFrame(rows).set_index("Model")
             .sort_values("Test AUC", ascending=False).round(3))
results

## 6. Which model ranks patients best?

In [ ]:
import plotly.express as px

chart = results.reset_index().sort_values("Test AUC")
fig = px.bar(chart, x="Test AUC", y="Model", orientation="h", text="Test AUC",
             title="Test ROC-AUC by model (higher is better, 0.5 = guessing)")
fig.update_traces(texttemplate="%{text:.3f}", marker_color="#1F5A85")
fig.update_xaxes(range=[0.5, 0.7])
fig.update_layout(height=500)
fig.show()

## 7. Overfitting check: Train AUC vs Test AUC
If the training score is much higher than the test score, the model memorised the training data.

In [ ]:
gap = results[["Train AUC", "Test AUC"]].reset_index().melt(
    id_vars="Model", var_name="Data", value_name="AUC")
fig = px.bar(gap, x="AUC", y="Model", color="Data", barmode="group", orientation="h",
             color_discrete_map={"Train AUC": "#8A99A8", "Test AUC": "#1F5A85"},
             title="Train vs test AUC (big gap = overfitting)")
fig.update_layout(height=600)
fig.show()

## 8. Analysis

1. **Boosting and Random Forest are the best group.** XGBoost, CatBoost, Random Forest, Gradient Boosting and LightGBM all reach a test AUC of about 0.67. They combine many small trees, which lets them find combinations of factors (for example: older patient + many past stays + sent to rehab).
2. **They are basically tied with each other.** I re-ran the top models on 5 different train/test splits. XGBoost and CatBoost averaged 0.667, LightGBM and Random Forest 0.666, Gradient Boosting 0.662. The differences between them are smaller than the change from one split to another.
3. **But they consistently beat Logistic Regression** (average 0.654) on every one of the 5 splits, by about 0.012. So the improvement is small but real.
4. **Overfitting is clear in two models.** The Decision Tree with no depth limit scored 1.000 on training data but only about 0.54 on test data: it memorised the patients. SVM did the same (about 0.97 train vs 0.57 test).
5. **KNN and SVM are a poor fit for this data.** KNN scored lowest of the normal models (about 0.63) and took several minutes just to predict. SVM had to be trained on a small sample because it is too slow on 80,000 patients, and it still performed worst.
6. **Recall is low for every model at the default cutoff**, because only 11% of patients are readmitted, so every model mostly predicts "No". This is why we compare models with ROC-AUC.
7. **Speed matters too.** XGBoost trained in about 2 seconds, while Random Forest took about 50 seconds for the same score.

## 9. Model choice: XGBoost

XGBoost is tied for the highest test AUC (about 0.675 on this split, 0.667 on average across 5 splits), beat Logistic Regression on every split, trains in about 2 seconds, and its train/test gap is small, so it is not badly overfitting. CatBoost is an equally good alternative.

**Logistic Regression is still a reasonable simple option:** it is only about 0.012 AUC behind, and it is much easier to explain because each factor gets one clear weight.

**Limitation:** all models land between 0.65 and 0.68 AUC, which suggests the limit comes from the data itself, not the model. Better results would need new information, such as lab values or social factors.

## 10. Save the best model (XGBoost) for the dashboard

During the comparison we trained on 80% of patients and tested on 20%. Now that XGBoost is chosen, we train it **one last time on all the data** and save it to `models/xgboost_model.joblib`.

The dashboard's **Risk predictor** tab loads this file. The saved file is the whole Pipeline (data preparation + XGBoost), so the app can give it raw patient details directly.

In [ ]:
import joblib
from xgboost import XGBClassifier

best_model = make_pipeline(XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=5,
                                         random_state=42, n_jobs=-1))
best_model.fit(X, y)                          # all patients this time

model_dir = Path.cwd().parent / "models"      # the models folder next to app.py
model_dir.mkdir(exist_ok=True)
joblib.dump(best_model, model_dir / "xgboost_model.joblib")
print("Saved:", model_dir / "xgboost_model.joblib")

## 11. Quick check: load the saved model and predict one patient

In [ ]:
loaded_model = joblib.load(model_dir / "xgboost_model.joblib")

one_patient = X.iloc[[0]]                     # double brackets keep it as a table
risk = loaded_model.predict_proba(one_patient)[0, 1]
print(f"Predicted 30-day readmission risk: {risk:.1%}")
one_patient